# Data visualisation

## Import libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scripts.hourly_forecasting.data_preparation.data_pipeline import prepare_data_forecasting
from scripts.hourly_forecasting.data_preparation.data_preparation import create_tz_column, datetime_conversion
from scripts.hourly_forecasting.data_preparation.data_cleaning import show_na, plot_na

## Load data

In [2]:
data_path = "../../../data/Hourly_Electricity_Demand_Gen_Weather_Spain/{}.csv";

energy_data_path = data_path.format("energy_dataset");
weather_data_path = data_path.format("weather_features");

In [3]:
energy_data = pd.read_csv(energy_data_path);
weather_data = pd.read_csv(weather_data_path);

## Run data pipeline

In [4]:
#new_energy_data = prepare_data_forecasting(energy_data);
#new_weather_data = prepare_data_forecasting(weather_data, "dt_iso");
new_energy_data = create_tz_column(energy_data, "time");
new_energy_data = datetime_conversion(new_energy_data, "time");
new_weather_data = create_tz_column(weather_data, "dt_iso");
new_weather_data = datetime_conversion(new_weather_data, "dt_iso");

In [5]:
#new_energy_data.join(new_weather_data, lsuffix="_energy", rsuffix="_weather", how="inner")
weather_df_list = [new_weather_data.groupby("city_name").get_group(city) for city in new_weather_data.city_name.unique().tolist()];

In [6]:
weather_df_list[0].loc[(~weather_df_list[0].index.duplicated(keep=False)).tolist()]

,city_name,temp,temp_min,temp_max,pressure,humidity,wind_speed,wind_deg,rain_1h,rain_3h,snow_3h,clouds_all,weather_id,weather_main,weather_description,weather_icon,tz_offset
dt_iso,,,,,,,,,,,,,,,,,
2014-12-31 23:00:00+00:00,Valencia,270.475,270.475,270.475,1001,77,1,62,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
2015-01-01 00:00:00+00:00,Valencia,270.475,270.475,270.475,1001,77,1,62,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
2015-01-01 01:00:00+00:00,Valencia,269.686,269.686,269.686,1002,78,0,23,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
2015-01-01 02:00:00+00:00,Valencia,269.686,269.686,269.686,1002,78,0,23,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
2015-01-01 03:00:00+00:00,Valencia,269.686,269.686,269.686,1002,78,0,23,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-12-31 18:00:00+00:00,Valencia,285.640,285.150,286.150,1028,62,2,140,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
2018-12-31 19:00:00+00:00,Valencia,283.140,282.150,284.150,1029,71,1,242,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
2018-12-31 20:00:00+00:00,Valencia,281.660,281.150,282.150,1029,81,3,300,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0


In [7]:
pd.to_datetime(weather_data.loc[weather_data["city_name"].isin(["Valencia"])].dt_iso, utc=True).duplicated(keep=False).sum()

162

In [8]:
weather_data.loc[(weather_data["city_name"].isin(["Valencia"])) & (weather_data["dt_iso"].str.contains("2018-12-13"))].head(12)

,dt_iso,city_name,temp,temp_min,temp_max,pressure,humidity,wind_speed,wind_deg,rain_1h,rain_3h,snow_3h,clouds_all,weather_id,weather_main,weather_description,weather_icon
34687,2018-12-13 00:00:00+01:00,Valencia,286.15,286.15,286.15,1015,66,2,0,0.0,0.0,0.0,20,801,clouds,few clouds,02n
34688,2018-12-13 01:00:00+01:00,Valencia,287.64,287.15,288.15,1014,58,4,260,0.0,0.0,0.0,20,801,clouds,few clouds,02n
34689,2018-12-13 02:00:00+01:00,Valencia,287.64,287.15,288.15,1014,54,6,280,0.0,0.0,0.0,20,801,clouds,few clouds,02n
34690,2018-12-13 03:00:00+01:00,Valencia,287.64,287.15,288.15,1013,54,8,280,0.0,0.0,0.0,20,801,clouds,few clouds,02n
34691,2018-12-13 04:00:00+01:00,Valencia,287.64,287.15,288.15,1013,58,5,260,0.0,0.0,0.0,20,801,clouds,few clouds,02n
34692,2018-12-13 05:00:00+01:00,Valencia,287.15,287.15,287.15,1012,62,5,270,0.0,0.0,0.0,20,801,clouds,few clouds,02n
34693,2018-12-13 06:00:00+01:00,Valencia,287.15,287.15,287.15,1011,67,7,280,0.0,0.0,0.0,20,801,clouds,few clouds,02n
34694,2018-12-13 07:00:00+01:00,Valencia,286.15,286.15,286.15,1011,76,2,220,0.3,0.0,0.0,20,500,rain,light rain,10n
34695,2018-12-13 08:00:00+01:00,Valencia,285.14,284.15,286.15,1011,76,4,270,0.3,0.0,0.0,40,500,rain,light rain,10n
34696,2018-12-13 08:00:00+01:00,Valencia,285.14,284.15,286.15,1011,76,4,270,0.3,0.0,0.0,40,701,mist,mist,50n


In [9]:
weather_data.loc[weather_data["city_name"].isin(["Valencia"])].dt_iso.duplicated(keep=False).sum()

162

In [10]:
(weather_df_list[0].index[weather_df_list[0].index.shape[0] - 1] - weather_df_list[0].index[0]).total_seconds() / 3600

35063.0

In [11]:
weather_df_list[0].loc[(~weather_df_list[0].index.duplicated()).tolist()].shape[0]

35064

In [12]:
weather_df_list[1].loc[(~weather_df_list[1].index.duplicated()).tolist()].shape[0]

35064

In [13]:
new_energy_data.columns

Index(['generation biomass', 'generation fossil brown coal/lignite',
       'generation fossil coal-derived gas', 'generation fossil gas',
       'generation fossil hard coal', 'generation fossil oil',
       'generation fossil oil shale', 'generation fossil peat',
       'generation geothermal', 'generation hydro pumped storage aggregated',
       'generation hydro pumped storage consumption',
       'generation hydro run-of-river and poundage',
       'generation hydro water reservoir', 'generation marine',
       'generation nuclear', 'generation other', 'generation other renewable',
       'generation solar', 'generation waste', 'generation wind offshore',
       'generation wind onshore', 'forecast solar day ahead',
       'forecast wind offshore eday ahead', 'forecast wind onshore day ahead',
       'total load forecast', 'total load actual', 'price day ahead',
       'price actual', 'tz_offset'],
      dtype='object')

In [14]:
pd.concat([weather_df_list[0].loc[~weather_df_list[0].index.duplicated()].copy(), weather_df_list[1].loc[~weather_df_list[1].index.duplicated()].copy()], axis=1).columns.tolist()

['city_name',
 'temp',
 'temp_min',
 'temp_max',
 'pressure',
 'humidity',
 'wind_speed',
 'wind_deg',
 'rain_1h',
 'rain_3h',
 'snow_3h',
 'clouds_all',
 'weather_id',
 'weather_main',
 'weather_description',
 'weather_icon',
 'tz_offset',
 'city_name',
 'temp',
 'temp_min',
 'temp_max',
 'pressure',
 'humidity',
 'wind_speed',
 'wind_deg',
 'rain_1h',
 'rain_3h',
 'snow_3h',
 'clouds_all',
 'weather_id',
 'weather_main',
 'weather_description',
 'weather_icon',
 'tz_offset']

In [15]:
weather_df_list[0].city_name.unique().item()

'Valencia'

In [16]:
prepare_data_forecasting(energy_data, weather_data)

,generation biomass,generation fossil brown coal/lignite,generation fossil coal-derived gas,generation fossil gas,generation fossil hard coal,generation fossil oil,generation fossil oil shale,generation fossil peat,generation geothermal,generation hydro pumped storage aggregated,...,wind_degSeville,rain_1hSeville,rain_3hSeville,snow_3hSeville,clouds_allSeville,weather_idSeville,weather_mainSeville,weather_descriptionSeville,weather_iconSeville,tz_offsetSeville
2014-12-31 23:00:00+00:00,447.0,329.0,0.0,4844.0,4821.0,162.0,0.0,0.0,0.0,NaN,...,21,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
2015-01-01 00:00:00+00:00,449.0,328.0,0.0,5196.0,4755.0,158.0,0.0,0.0,0.0,NaN,...,21,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
2015-01-01 01:00:00+00:00,448.0,323.0,0.0,4857.0,4581.0,157.0,0.0,0.0,0.0,NaN,...,27,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
2015-01-01 02:00:00+00:00,438.0,254.0,0.0,4314.0,4131.0,160.0,0.0,0.0,0.0,NaN,...,27,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
2015-01-01 03:00:00+00:00,428.0,187.0,0.0,4130.0,3840.0,156.0,0.0,0.0,0.0,NaN,...,27,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-12-31 18:00:00+00:00,297.0,0.0,0.0,7634.0,2628.0,178.0,0.0,0.0,0.0,NaN,...,30,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
2018-12-31 19:00:00+00:00,296.0,0.0,0.0,7241.0,2566.0,174.0,0.0,0.0,0.0,NaN,...,30,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
2018-12-31 20:00:00+00:00,292.0,0.0,0.0,7025.0,2422.0,168.0,0.0,0.0,0.0,NaN,...,50,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
2018-12-31 21:00:00+00:00,293.0,0.0,0.0,6562.0,2293.0,163.0,0.0,0.0,0.0,NaN,...,60,0.0,0.0,0.0,0,800,clear,sky is clear,01n,1.0
